# Apache Log Parsing with Semiformal Programming

This notebook demonstrates how semiformal programming bridges formal and informal
logic in a realistic log parsing pipeline. The formal parts (prefix regex, compiled
parser) stay deterministic. The semiformal parts (`semi()` calls) handle deployment-
specific decisions: body classification and regex template generation.

Key behaviors demonstrated:
- **GENERATE**: First invocation creates an implementation via the agentic pipeline
- **REUSE**: Subsequent calls with the same template reuse the cached implementation
- **Edge cases**: New patterns surface as `UNSEEN_TEMPLATE` -- the parser does not guess
- **Wider bootstrap**: Updated prompt context triggers new generation for broader coverage

In [1]:
import shutil
from pathlib import Path

cache = Path(".semiformal")
if cache.exists():
    shutil.rmtree(cache)
    print(f"Cleared {cache}")
else:
    print("No cache to clear")

from semipy import configure, semi

configure(verbose=True)

No cache to clear


## The Raw Problem

Apache error logs are human-readable but downstream systems (dashboards, anomaly
detection, routing) need structured events. Each deployment has a slightly different
message ecology. We want to **compile** a parser from a local bootstrap slice, then
reuse it deterministically for all future logs.

In [2]:
import re
from collections import Counter

LOG_PATH = Path("data/Apache_2k.log")
all_lines = [ln for ln in LOG_PATH.read_text().splitlines() if ln.strip()]

print(f"Total lines: {len(all_lines)}\n")
print("First 8 lines:")
for line in all_lines[:8]:
    print(f"  {line}")
print("  ...")
print("Last 5 lines:")
for line in all_lines[-5:]:
    print(f"  {line}")

Total lines: 2000

First 8 lines:
  [Sun Dec 04 04:47:44 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
  [Sun Dec 04 04:47:44 2005] [error] mod_jk child workerEnv in error state 6
  [Sun Dec 04 04:51:08 2005] [notice] jk2_init() Found child 6725 in scoreboard slot 10
  [Sun Dec 04 04:51:09 2005] [notice] jk2_init() Found child 6726 in scoreboard slot 8
  [Sun Dec 04 04:51:09 2005] [notice] jk2_init() Found child 6728 in scoreboard slot 6
  [Sun Dec 04 04:51:14 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
  [Sun Dec 04 04:51:14 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
  [Sun Dec 04 04:51:14 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties
  ...
Last 5 lines:
  [Mon Dec 05 19:14:11 2005] [error] mod_jk child workerEnv in error state 6
  [Mon Dec 05 19:15:55 2005] [notice] jk2_init() Found child 6791 in scoreboard slot 8
  [Mon Dec 05 19:15:55 2005] [notice] jk2_init() Found child 6790 i

## Stage 1: Formal Prefix Parsing

The log format has a known structure: `[timestamp] [level] body`. We parse this
formally with a regex. No LLM needed -- this is ordinary software engineering.
Only the **body** patterns are deployment-specific.

In [3]:
APACHE_PREFIX = re.compile(
    r"^\[(?P<ts>[^\]]+)\]\s+\[(?P<level>[^\]]+)\]\s+(?P<body>.*)$"
)

def parse_prefix(line):
    m = APACHE_PREFIX.match(line.strip())
    return m.groupdict() if m else None

bootstrap = all_lines[:120]
parsed = [p for ln in bootstrap if (p := parse_prefix(ln)) is not None]
bodies = [p["body"] for p in parsed]
unique_bodies = sorted(set(bodies))

print(f"Bootstrap: {len(bootstrap)} lines")
print(f"Parsed: {len(parsed)} (prefix match rate: {len(parsed)/len(bootstrap):.0%})")
print(f"Unique body patterns: {len(unique_bodies)}\n")
for b in unique_bodies:
    print(f"  {b}")

Bootstrap: 120 lines
Parsed: 120 (prefix match rate: 100%)
Unique body patterns: 50

  jk2_init() Found child 25792 in scoreboard slot 6
  jk2_init() Found child 6725 in scoreboard slot 10
  jk2_init() Found child 6726 in scoreboard slot 8
  jk2_init() Found child 6728 in scoreboard slot 6
  jk2_init() Found child 6733 in scoreboard slot 7
  jk2_init() Found child 6734 in scoreboard slot 9
  jk2_init() Found child 6736 in scoreboard slot 10
  jk2_init() Found child 6737 in scoreboard slot 8
  jk2_init() Found child 6738 in scoreboard slot 6
  jk2_init() Found child 6740 in scoreboard slot 7
  jk2_init() Found child 6741 in scoreboard slot 9
  jk2_init() Found child 6744 in scoreboard slot 10
  jk2_init() Found child 6745 in scoreboard slot 8
  jk2_init() Found child 6748 in scoreboard slot 6
  jk2_init() Found child 6750 in scoreboard slot 7
  jk2_init() Found child 6751 in scoreboard slot 9
  jk2_init() Found child 6752 in scoreboard slot 10
  jk2_init() Found child 6754 in scoreboard

## Stage 2: Semiformal Body Classification

The body messages are deployment-specific. We use `semi()` to classify each unique
body into an event family. The **first call** triggers **GENERATE** -- the agentic
pipeline creates a classification function. All subsequent calls at the same call
site **REUSE** the cached implementation (no LLM invocation).

In [4]:
family_map = {}
for body in unique_bodies:
    family = semi(
        f"Classify this Apache error log body into a short snake_case event family name: {body}"
    )
    family_map[body] = family

print("Body -> Family mapping:\n")
for body, family in family_map.items():
    print(f"  {body[:60]:60s} -> {family}")

families = sorted(set(family_map.values()))
print(f"\nDiscovered {len(families)} families: {families}")

Output()

Body -> Family mapping:

  jk2_init() Found child 25792 in scoreboard slot 6            -> worker_initialization
  jk2_init() Found child 6725 in scoreboard slot 10            -> worker_initialization
  jk2_init() Found child 6726 in scoreboard slot 8             -> worker_initialization
  jk2_init() Found child 6728 in scoreboard slot 6             -> worker_initialization
  jk2_init() Found child 6733 in scoreboard slot 7             -> worker_initialization
  jk2_init() Found child 6734 in scoreboard slot 9             -> worker_initialization
  jk2_init() Found child 6736 in scoreboard slot 10            -> worker_initialization
  jk2_init() Found child 6737 in scoreboard slot 8             -> worker_initialization
  jk2_init() Found child 6738 in scoreboard slot 6             -> worker_initialization
  jk2_init() Found child 6740 in scoreboard slot 7             -> worker_initialization
  jk2_init() Found child 6741 in scoreboard slot 9             -> worker_initialization
  jk2_i

## REUSE: Same Implementation, New Data

Calling `semi()` with the same template structure from a **different cell** triggers
cross-slot REUSE via equivalence-key matching. The `spec_equivalence_key` (based on
template text and variable names, not runtime values) matches the donor slot above.

Notice: no generation UI appears -- the cached function is loaded instantly.

In [5]:
test_bodies = [
    "workerEnv.init() ok /etc/httpd/conf/workers2.properties",
    "mod_jk child workerEnv in error state 6",
    "jk2_init() Found child 99999 in scoreboard slot 42",
]

print("Re-classifying from a new cell (should REUSE via donor):\n")
for body in test_bodies:
    family = semi(
        f"Classify this Apache error log body into a short snake_case event family name: {body}"
    )
    print(f"  {body[:60]:60s} -> {family}")

Re-classifying from a new cell (should REUSE via donor):

  workerEnv.init() ok /etc/httpd/conf/workers2.properties      -> workerenv_init_ok_etc
  mod_jk child workerEnv in error state 6                      -> jk_child_workerenv_error
  jk2_init() Found child 99999 in scoreboard slot 42           -> worker_initialization


## Stage 3: Semiformal Template Generation

For each event family, we generate a regex template with named capture groups.
This is the second semiformal decision in the pipeline: the agent sees the family
name and example bodies, and produces a pattern that matches the entire body string.

In [6]:
import json

grouped = {}
for body, fam in family_map.items():
    grouped.setdefault(fam, []).append(body)

families_text = ""
for fam, fam_bodies in sorted(grouped.items()):
    samples = "\n".join(f"    - {b}" for b in sorted(set(fam_bodies))[:15])
    families_text += f"\n  {fam}:\n{samples}"

templates = semi(
    f"For each event family, create a Python regex that matches the entire body string. "
    f"Use named capture groups for variable parts only (keep stable tokens as literals). "
    f"Return a list of dicts with keys: family (str), pattern (str), "
    f"fields (dict mapping capture group name to type hint like 'int' or 'str').\n"
    f"Families and example bodies:{families_text}",
    expected_type=list,
)

print(f"Generated {len(templates)} templates:\n")
for t in templates:
    print(f"  {t.get('family', '?')}:")
    print(f"    pattern: {t.get('pattern', '?')}")
    print(f"    fields:  {t.get('fields', {})}")

Output()

Generated 3 templates:

  jk_child_workerenv_error:
    pattern: ^mod_jk\ child\ workerEnv\ in\ error\ state\ (?P<field1>\d+)$
    fields:  {'field1': 'int'}
  worker_initialization:
    pattern: ^jk2_init\(\)\ Found\ child\ (?P<field1>\d+)\ in\ scoreboard\ slot\ (?P<field2>\d+)$
    fields:  {'field1': 'int', 'field2': 'int'}
  workerenv_init_ok_etc:
    pattern: ^workerEnv\.init\(\)\ ok\ /etc/httpd/conf/workers2\.properties$
    fields:  {}


## Stage 4: Compiled Deterministic Parser

The semiformal templates are frozen into a deterministic regex parser. From here on,
parsing is pure formal execution -- no LLM calls per line. This is the key property:
the inferential work happens once at compile time, and the result is a reusable artifact.

In [7]:
rules = []
for t in templates:
    try:
        pat = re.compile(t["pattern"])
        rules.append((t["family"], pat, t.get("fields", {})))
        print(f"  Compiled: {t['family']}")
    except re.error as e:
        print(f"  INVALID regex for {t['family']}: {e}")

print(f"\nParsing {len(all_lines)} lines with {len(rules)} rules...\n")

results = []
for line in all_lines:
    prefix = parse_prefix(line)
    if not prefix:
        results.append({"status": "PREFIX_FAIL"})
        continue
    body = prefix["body"]
    matched = False
    for fam, pat, fields in rules:
        m = pat.match(body)
        if m:
            results.append({"status": "OK", "family": fam, "body": body,
                            "captures": {k: v for k, v in m.groupdict().items() if v is not None}})
            matched = True
            break
    if not matched:
        results.append({"status": "UNSEEN_TEMPLATE", "body": body})

counts = Counter(r["status"] for r in results)
print("Status counts:")
for status, count in counts.most_common():
    print(f"  {status}: {count}")

unseen = sorted(set(r.get("body", "") for r in results if r["status"] == "UNSEEN_TEMPLATE"))
if unseen:
    print(f"\nUnseen body patterns ({len(unseen)} distinct):")
    for b in unseen:
        print(f"  {b!r}")

ok_families = Counter(r["family"] for r in results if r["status"] == "OK")
print(f"\nFamily distribution (OK lines):")
for fam, count in ok_families.most_common():
    print(f"  {fam}: {count}")

  Compiled: jk_child_workerenv_error
  Compiled: worker_initialization
  Compiled: workerenv_init_ok_etc

Parsing 2000 lines with 3 rules...

Status counts:
  OK: 1944
  UNSEEN_TEMPLATE: 56

Unseen body patterns (45 distinct):
  '[client 125.30.38.52] Directory index forbidden by rule: /var/www/html/'
  '[client 141.153.150.164] Directory index forbidden by rule: /var/www/html/'
  '[client 141.154.18.244] Directory index forbidden by rule: /var/www/html/'
  '[client 147.31.138.75] Directory index forbidden by rule: /var/www/html/'
  '[client 168.20.198.21] Directory index forbidden by rule: /var/www/html/'
  '[client 198.232.168.9] Directory index forbidden by rule: /var/www/html/'
  '[client 207.12.15.211] Directory index forbidden by rule: /var/www/html/'
  '[client 207.203.80.15] Directory index forbidden by rule: /var/www/html/'
  '[client 208.51.151.210] Directory index forbidden by rule: /var/www/html/'
  '[client 211.141.93.88] Directory index forbidden by rule: /var/www/html/'


## Runtime Context Does Not Trigger Recompilation

Changing deployment labels, host names, or which day's log file you parse does not
change the parser. Same compiled artifact, different metadata. This is a key property
of semiformal programming: the inferential work is done once, and runtime variation
flows through the formal pipeline without touching the `semi()` slots.

In [8]:
print("Same parser, different runtime context (no LLM calls):\n")
for host in ["web-01", "web-02", "staging-01"]:
    host_results = []
    for line in all_lines[:300]:
        prefix = parse_prefix(line)
        if not prefix:
            continue
        body = prefix["body"]
        for fam, pat, fields in rules:
            if pat.match(body):
                host_results.append({"host": host, "family": fam})
                break
    family_counts = Counter(r["family"] for r in host_results)
    print(f"  host={host}: {dict(family_counts)}")

Same parser, different runtime context (no LLM calls):

  host=web-01: {'workerenv_init_ok_etc': 93, 'jk_child_workerenv_error': 88, 'worker_initialization': 118}
  host=web-02: {'workerenv_init_ok_etc': 93, 'jk_child_workerenv_error': 88, 'worker_initialization': 118}
  host=staging-01: {'workerenv_init_ok_etc': 93, 'jk_child_workerenv_error': 88, 'worker_initialization': 118}


## Edge Cases: New Patterns Arrive

A new Apache deployment or module variant sends logs with patterns the bootstrap
never saw. The parser correctly reports them as `UNSEEN_TEMPLATE` -- it does not
guess. This is the healthy failure mode.

We also add synthetic edge cases representing common Apache error patterns from
other deployments (client errors, SSL failures, bind errors).

In [9]:
edge_cases = [
    "[Wed Dec 07 12:00:00 2005] [error] [client 192.168.1.100] File does not exist: /var/www/html/favicon.ico",
    "[Wed Dec 07 12:01:00 2005] [crit] (98)Address already in use: make_sock: could not bind to address 0.0.0.0:80",
    "[Wed Dec 07 12:02:00 2005] [warn] mod_ssl: SSL handshake failed for client 10.0.0.5",
    "[Wed Dec 07 12:03:00 2005] [error] [client 172.16.0.1] Invalid URI in request GET /../../etc/passwd HTTP/1.0",
    "[Wed Dec 07 12:04:00 2005] [notice] Apache/2.0.54 configured -- resuming normal operations",
]

print("Edge case bodies through the narrow parser:\n")
for line in edge_cases:
    prefix = parse_prefix(line)
    if not prefix:
        print(f"  PREFIX_FAIL: {line[:80]}")
        continue
    body = prefix["body"]
    matched = False
    for fam, pat, fields in rules:
        if pat.match(body):
            print(f"  OK [{fam}]: {body[:80]}")
            matched = True
            break
    if not matched:
        print(f"  UNSEEN_TEMPLATE: {body[:80]}")

print("\nAlso check bodies that appear later in the log but not in the bootstrap:")
later_bodies = sorted(set(
    p["body"] for ln in all_lines[120:] if (p := parse_prefix(ln)) is not None
) - set(unique_bodies))
for b in later_bodies[:10]:
    matched = any(pat.match(b) for _, pat, _ in rules)
    status = "OK" if matched else "UNSEEN_TEMPLATE"
    print(f"  [{status}] {b[:80]}")

Edge case bodies through the narrow parser:

  UNSEEN_TEMPLATE: [client 192.168.1.100] File does not exist: /var/www/html/favicon.ico
  UNSEEN_TEMPLATE: (98)Address already in use: make_sock: could not bind to address 0.0.0.0:80
  UNSEEN_TEMPLATE: mod_ssl: SSL handshake failed for client 10.0.0.5
  UNSEEN_TEMPLATE: [client 172.16.0.1] Invalid URI in request GET /../../etc/passwd HTTP/1.0
  UNSEEN_TEMPLATE: Apache/2.0.54 configured -- resuming normal operations

Also check bodies that appear later in the log but not in the bootstrap:
  [UNSEEN_TEMPLATE] [client 125.30.38.52] Directory index forbidden by rule: /var/www/html/
  [UNSEEN_TEMPLATE] [client 141.153.150.164] Directory index forbidden by rule: /var/www/html/
  [UNSEEN_TEMPLATE] [client 141.154.18.244] Directory index forbidden by rule: /var/www/html/
  [UNSEEN_TEMPLATE] [client 147.31.138.75] Directory index forbidden by rule: /var/www/html/
  [UNSEEN_TEMPLATE] [client 168.20.198.21] Directory index forbidden by rule: /var/www/

## Extending the Parser: New Families from Unseen Patterns

The narrow parser correctly reported `UNSEEN_TEMPLATE` for patterns it does not know.
The engineer now inspects those failures and extends the parser:

1. **Batch-classify unseen bodies** into new event families (one `semi()` call)
2. **Generate regex templates** for the new families (REUSE from the template generator)
3. **Merge** narrow + new rules into an extended parser
4. **Re-run** on the full dataset

The formal prefix parsing and compiled parser logic stay untouched. Only the
inferential decisions (classification and template generation) are extended.

In [10]:
extended_lines = all_lines + edge_cases

unseen_bodies_from_parser = sorted(set(
    r.get("body", "") for r in results if r["status"] == "UNSEEN_TEMPLATE"
))
unseen_from_edges = [
    parse_prefix(ln)["body"] for ln in edge_cases if parse_prefix(ln)
]
all_unseen = sorted(set(unseen_bodies_from_parser + unseen_from_edges))

print(f"Unseen patterns to classify ({len(all_unseen)} distinct):\n")
for b in all_unseen[:10]:
    print(f"  {b}")
if len(all_unseen) > 10:
    print(f"  ... and {len(all_unseen) - 10} more")

Unseen patterns to classify (50 distinct):

  (98)Address already in use: make_sock: could not bind to address 0.0.0.0:80
  Apache/2.0.54 configured -- resuming normal operations
  [client 125.30.38.52] Directory index forbidden by rule: /var/www/html/
  [client 141.153.150.164] Directory index forbidden by rule: /var/www/html/
  [client 141.154.18.244] Directory index forbidden by rule: /var/www/html/
  [client 147.31.138.75] Directory index forbidden by rule: /var/www/html/
  [client 168.20.198.21] Directory index forbidden by rule: /var/www/html/
  [client 172.16.0.1] Invalid URI in request GET /../../etc/passwd HTTP/1.0
  [client 192.168.1.100] File does not exist: /var/www/html/favicon.ico
  [client 198.232.168.9] Directory index forbidden by rule: /var/www/html/
  ... and 40 more


In [ ]:
unseen_text = "\n".join(f"  - {b}" for b in all_unseen)

new_families = semi(
    f"Group these Apache log body messages into event families. "
    f"Return a dict mapping short snake_case family name to a list of body strings "
    f"that belong to it.\n"
    f"Bodies:\n{unseen_text}",
    expected_type=dict,
)

print(f"New families discovered ({len(new_families)}):\n")
for fam, fam_bodies in sorted(new_families.items()):
    print(f"  {fam} ({len(fam_bodies)} bodies):")
    for b in fam_bodies[:3]:
        print(f"    {b}")
    if len(fam_bodies) > 3:
        print(f"    ... and {len(fam_bodies) - 3} more")

In [ ]:
new_families_text = ""
for fam, fam_bodies in sorted(new_families.items()):
    samples = "\n".join(f"    - {b}" for b in sorted(set(fam_bodies))[:15])
    new_families_text += f"\n  {fam}:\n{samples}"

new_templates = semi(
    f"For each event family, create a Python regex that matches the entire body string. "
    f"Use named capture groups for variable parts only (keep stable tokens as literals). "
    f"Each pattern MUST be specific to its family -- never use catch-all patterns like '(.*)'. "
    f"Return a list of dicts with keys: family (str), pattern (str), "
    f"fields (dict mapping capture group name to type hint like 'int' or 'str').\n"
    f"Families and example bodies:{new_families_text}",
    expected_type=list,
)


print(f"Generated {len(new_templates)} new templates:\n")
for t in new_templates:
    print(f"  {t.get('family', '?')}: {t.get('pattern', '?')[:70]}")

In [ ]:
extended_rules = list(rules)
for t in new_templates:
    pat_str = t.get("pattern", "")
    if pat_str in ("^(?P<body>.*)$", "^(.*)$", ".*", "^.*$"):
        print(f"  SKIPPED catch-all: {t.get('family', '?')}")
        continue
    try:
        pat = re.compile(pat_str)
        extended_rules.append((t["family"], pat, t.get("fields", {})))
        print(f"  Added: {t['family']}")
    except re.error as e:
        print(f"  INVALID regex for {t.get('family', '?')}: {e}")

print(f"\nExtended parser: {len(rules)} narrow + {len(extended_rules) - len(rules)} new = {len(extended_rules)} rules\n")

ext_results = []
for line in extended_lines:
    prefix = parse_prefix(line)
    if not prefix:
        ext_results.append({"status": "PREFIX_FAIL"})
        continue
    body = prefix["body"]
    matched = False
    for fam, pat, fields in extended_rules:
        m = pat.match(body)
        if m:
            ext_results.append({"status": "OK", "family": fam, "body": body})
            matched = True
            break
    if not matched:
        ext_results.append({"status": "UNSEEN_TEMPLATE", "body": body})

ext_counts = Counter(r["status"] for r in ext_results)
print("Extended parser status counts:")
for status, count in ext_counts.most_common():
    print(f"  {status}: {count}")

ext_unseen = sorted(set(r.get("body", "") for r in ext_results if r["status"] == "UNSEEN_TEMPLATE"))
if ext_unseen:
    print(f"\nStill unseen ({len(ext_unseen)} distinct):")
    for b in ext_unseen:
        print(f"  {b!r}")

narrow_ok = counts.get("OK", 0)
ext_ok = ext_counts.get("OK", 0)
print(f"\n--- Coverage comparison ---")
print(f"Narrow parser: {narrow_ok}/{len(all_lines)} lines OK ({narrow_ok/len(all_lines):.1%})")
print(f"Extended parser: {ext_ok}/{len(extended_lines)} lines OK ({ext_ok/len(extended_lines):.1%})")

ext_fam_counts = Counter(r["family"] for r in ext_results if r["status"] == "OK")
print(f"\nFamily distribution (extended parser):")
for fam, count in ext_fam_counts.most_common():
    print(f"  {fam}: {count}")

## Summary

What happened:

1. **Formal prefix parsing** separated timestamp, level, and body -- no LLM needed.
2. **Semiformal classification** created a body classifier from a bootstrap slice (GENERATE),
   then reused it instantly for new data (REUSE).
3. **Semiformal template generation** created regex patterns per family (GENERATE),
   then reused the same template generator for new families (REUSE).
4. **Formal compiled parser** ran deterministically on 2000+ lines -- no LLM per line.
5. **Edge cases** were correctly reported as UNSEEN_TEMPLATE -- the parser does not guess.
6. **Extension**: unseen patterns were batch-classified into new families, templates were
   generated, and the parser was extended -- all using the same semiformal pipeline.

The invariant did not change: one raw line becomes one conservative structured event.
What changed is the local execution logic for newly observed families. The formal
pipeline (prefix regex, compiled parser, batch execution) stayed untouched throughout.

Key semiformal properties demonstrated:
- **Compile once, execute many**: LLM calls happen at compile time, not per line.
- **Equivalence-based REUSE**: Same template structure = same cached implementation.
- **Conservative failure**: Unknown patterns surface as structured failure categories.
- **Incremental extension**: New families extend the parser without replacing working rules.